# Novel jailbreak detection

For each evaluated type, every prompt with that label is held out of training.
Jailbreak and non-jailbreak prompts go through the same stemmer and stop-word
list. Non-jailbreak prompts are drawn from one permutation and then cut into a
training block and a test block, so the two sides are disjoint.

`NON_JAILBREAK_RATIO` is the number of non-jailbreak prompts per jailbreak
prompt, on each side. The historical notebooks hardcoded 10, which is higher
than the binary corpus (about 1.4 non-jailbreaks per jailbreak). 10 stays the
default so a rerun is comparable; change the constant to match another
protocol. `ratio_matching_corpus` computes the source-corpus ratio.

The split audit is written even when `RUN_TRAINING=0`. The fine-tune itself is
skipped in that mode.

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

import json

import torch
import pandas as pd
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments

from labels import EVALUATED_JAILBREAK_TYPES
from latex_tables import export_results
from metrics import trainer_compute_metrics
from novel_splits import (
    DEFAULT_NON_JAILBREAK_RATIO,
    build_heldout_splits,
    ratio_matching_corpus,
)
from paths import data_path, results_path
from text_preprocess import stem_preprocess

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Non-jailbreak prompts per jailbreak prompt. 10 is the historical setting.
NON_JAILBREAK_RATIO = float(os.environ.get("NON_JAILBREAK_RATIO", DEFAULT_NON_JAILBREAK_RATIO))
RUN_TRAINING = os.environ.get("RUN_TRAINING", "1") == "1"

jailbreak_df = pd.read_csv(data_path("all_labelled_cleaned.csv"))
regular_df = pd.read_csv(data_path("regular_prompts_DAN.csv"))
print(
    "Source ratio of non-jailbreak to jailbreak prompts:",
    round(ratio_matching_corpus(len(regular_df), len(jailbreak_df)), 3),
)
print("Using NON_JAILBREAK_RATIO =", NON_JAILBREAK_RATIO)

splits = {}
audit = []
for label in EVALUATED_JAILBREAK_TYPES:
    train_df, test_df, info = build_heldout_splits(
        jailbreak_df,
        regular_df,
        label,
        ratio=NON_JAILBREAK_RATIO,
        random_state=1,
        preprocess_fn=stem_preprocess,
    )
    overlap = set(train_df["prompt"]) & set(test_df["prompt"])
    if overlap:
        raise RuntimeError(f"{label} leaked {len(overlap)} prompts across the split")
    splits[label] = (train_df, test_df)
    audit.append(info.as_dict())
    print(
        f"{label}: jailbreak train/test {info.n_train_jailbreak}/{info.n_test_jailbreak}, "
        f"regular train/test {info.n_train_regular}/{info.n_test_regular}, overlap={info.prompt_overlap}"
    )

audit_path = results_path("07_novel_split_audit.json")
audit_path.write_text(json.dumps({"ratio": NON_JAILBREAK_RATIO, "splits": audit}, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {audit_path}")

class PromptDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        self.prompts = frame["prompt"].tolist()
        self.labels = frame["Jailbreak"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            str(self.prompts[index]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze() for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

def make_training_args(**kwargs):
    try:
        return TrainingArguments(**kwargs)
    except TypeError:
        if "eval_strategy" in kwargs:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
            return TrainingArguments(**kwargs)
        raise

if RUN_TRAINING:
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    rows = []
    for label, (train_df, test_df) in splits.items():
        print(f"\nTraining with {label} held out")
        output_dir = ROOT / "models" / "novel_jailbreak_bert" / label.lower().replace(" ", "_")
        output_dir.mkdir(parents=True, exist_ok=True)
        model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
        trainer = Trainer(
            model=model,
            args=make_training_args(
                output_dir=str(output_dir / "checkpoints"),
                num_train_epochs=4,
                per_device_train_batch_size=8,
                per_device_eval_batch_size=8,
                warmup_steps=10,
                weight_decay=0.01,
                logging_dir=str(output_dir / "logs"),
                logging_steps=10,
                eval_strategy="epoch",
                save_strategy="no",
                report_to=[],
            ),
            train_dataset=PromptDataset(train_df, tokenizer),
            eval_dataset=PromptDataset(test_df, tokenizer),
            compute_metrics=trainer_compute_metrics,
        )
        trainer.train()
        trainer.save_model(str(output_dir))
        evaluation = trainer.evaluate()
        rows.append(
            {
                "held_out_type": label,
                "auc": evaluation["eval_auc"],
                "accuracy": evaluation["eval_accuracy"],
                "fnr": evaluation["eval_fnr"],
                "tpr": evaluation["eval_tpr"],
                "f1": evaluation["eval_f1"],
            }
        )
        print(rows[-1])
    paths = export_results("07_novel_jailbreak", "novel", rows)
    print(f"Wrote {paths['tex']}")
else:
    print("RUN_TRAINING=0, so no novel-jailbreak model was fit.")